In [2]:
import pandas as pd

In [3]:
data = pd.read_csv('../data/data_globant.csv')

In [4]:
data.head()

,Date,Email,Name,Position,Seniority,Location,Studio,Client,Client Tag,Project,Project Tag,Team Name,Engagement,Email Leader,Year,Month,Day
0,02Jan23,natalia.ramirez@tec.globant.com,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GreenWave Innovations,GWI001,Atlas Initiative,ATLINT,Breaking Badger,3.04,laura.leon@tec.globant.com,2023,1,2
1,03Jan23,natalia.ramirez@tec.globant.com,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GreenWave Innovations,GWI001,Atlas Initiative,ATLINT,Breaking Badger,2.99,laura.leon@tec.globant.com,2023,1,3
2,04Jan23,natalia.ramirez@tec.globant.com,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GreenWave Innovations,GWI001,Atlas Initiative,ATLINT,Breaking Badger,2.97,laura.leon@tec.globant.com,2023,1,4
3,05Jan23,natalia.ramirez@tec.globant.com,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GreenWave Innovations,GWI001,Atlas Initiative,ATLINT,Breaking Badger,2.75,laura.leon@tec.globant.com,2023,1,5
4,06Jan23,natalia.ramirez@tec.globant.com,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GreenWave Innovations,GWI001,Atlas Initiative,ATLINT,Breaking Badger,3.15,laura.leon@tec.globant.com,2023,1,6


### Date to timestamp

In [5]:
data['Date'] = pd.to_datetime(data['Date'], format='%d%b%y')

In [6]:
data['Date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 11366 entries, 0 to 11365
Series name: Date
Non-Null Count  Dtype         
--------------  -----         
11366 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 88.9 KB


### Drop columns

In [7]:
data.drop(columns = ['Email', 'Client','Project', 'Year', 'Month', 'Day'], inplace = True)

In [8]:
data.head()

,Date,Name,Position,Seniority,Location,Studio,Client Tag,Project Tag,Team Name,Engagement,Email Leader
0,2023-01-02,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.04,laura.leon@tec.globant.com
1,2023-01-03,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.99,laura.leon@tec.globant.com
2,2023-01-04,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.97,laura.leon@tec.globant.com
3,2023-01-05,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.75,laura.leon@tec.globant.com
4,2023-01-06,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.15,laura.leon@tec.globant.com


### formating email leader

In [9]:
data['Email Leader'] = data['Email Leader'].str.split('@').str[0]
data['Email Leader'] = data['Email Leader'].str.replace(r'[._]', ' ', regex=True).str.title()

In [10]:
data = data.rename(columns={'Email Leader': 'Leader Name'})

In [11]:
data.head()

,Date,Name,Position,Seniority,Location,Studio,Client Tag,Project Tag,Team Name,Engagement,Leader Name
0,2023-01-02,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.04,Laura Leon
1,2023-01-03,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.99,Laura Leon
2,2023-01-04,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.97,Laura Leon
3,2023-01-05,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.75,Laura Leon
4,2023-01-06,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.15,Laura Leon


### Name leader for managers

In [12]:
data['Leader Name'] = data['Leader Name'].fillna(data['Name'])

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11366 entries, 0 to 11365
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         11366 non-null  datetime64[ns]
 1   Name         11366 non-null  object        
 2   Position     11366 non-null  object        
 3   Seniority    11366 non-null  object        
 4   Location     11366 non-null  object        
 5   Studio       11366 non-null  object        
 6   Client Tag   11366 non-null  object        
 7   Project Tag  11366 non-null  object        
 8   Team Name    11366 non-null  object        
 9   Engagement   11366 non-null  float64       
 10  Leader Name  11366 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(9)
memory usage: 976.9+ KB


### Deteccion de duplicados

In [14]:
data.duplicated().sum()

0

### Valores invalidos

In [15]:
data[(data['Engagement'] < 0) | (data['Engagement'] > 5)].shape[0]

0

### Engagement a discreto

In [16]:
# 1. Creamos una serie nueva con puros 0 como base
groups = pd.Series(0, index=data.index, dtype="int64")

# 2. Máscara para valores > 0
mask_pos = data['Engagement'] > 0

# 3. Aplicamos quintiles solo a los > 0
quintiles = pd.qcut(
    data.loc[mask_pos, 'Engagement'],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

# 4. Asignamos esos quintiles a la serie 'groups'
groups.loc[mask_pos] = quintiles.astype(int)

# 5. Guardamos el resultado en una nueva columna del DataFrame
data['Engagement Group'] = groups

In [17]:
data.head(35)

,Date,Name,Position,Seniority,Location,Studio,Client Tag,Project Tag,Team Name,Engagement,Leader Name,Engagement Group
0,2023-01-02,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.04,Laura Leon,3
1,2023-01-03,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.99,Laura Leon,3
2,2023-01-04,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.97,Laura Leon,3
3,2023-01-05,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.75,Laura Leon,2
4,2023-01-06,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.15,Laura Leon,3
5,2023-01-09,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.47,Laura Leon,4
6,2023-01-10,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.20,Laura Leon,3
7,2023-01-11,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,2.70,Laura Leon,2
8,2023-01-12,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.31,Laura Leon,4
9,2023-01-13,Natalia Ramírez,Software Developer,Jr,CO/ANT/MED,Engineering,GWI001,ATLINT,Breaking Badger,3.04,Laura Leon,3


In [18]:
data['Engagement Group'].value_counts().sort_index()

Engagement Group
0     268
1    2269
2    2186
3    2246
4    2179
5    2218
Name: count, dtype: int64

### exportar csv

In [19]:
data.to_csv('../data/data_globant_cleaned.csv', index=False)